In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Setup working directory in Google Drive
import os
assert os.path.exists('/content/drive')
WORK_DIR = '/content/drive/MyDrive/colab/tiger_semantic_id'
%mkdir -p $WORK_DIR

In [ ]:
# Clean and create data preparation folder
import shutil
import os

DATA_PREP_DIR = WORK_DIR + '/data_preparation'

# Remove existing folder if it exists
if os.path.exists(DATA_PREP_DIR):
    print(f"Removing existing {DATA_PREP_DIR} folder...")
    shutil.rmtree(DATA_PREP_DIR)

# Create fresh folder
os.makedirs(DATA_PREP_DIR)
print(f"✅ Created fresh {DATA_PREP_DIR} folder")

# TIGER SemanticID — Data Preparation

Goal: Implement Semantic IDs via RQ-VAE and a compact seq2seq Transformer for generative retrieval on Amazon 5-core datasets; produce metrics and visualizations validating paper claims.

Datasets: Amazon Product Reviews (Beauty, Video_Games). Switch between datasets using the `dataset_name` config parameter.

Key steps: Download & preprocess; Sentence-T5 embeddings; RQ-VAE (3 levels, K=256) to 3-tuple codes + collision code c4; visualizations (c1↔category, hierarchy); seq2seq generative retrieval; metrics Recall@5/10, NDCG@5/10 and invalid-ID rate; ablations (Random/LSH); mini cold-start probe.

Artifacts: save to /content/artifacts. Keep configs modest for Colab; add knobs for smoke tests.

In [ ]:
# Clone repo, install dependencies, and make src importable (Colab-friendly)
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

repo_url = 'https://github.com/allyoushawn/recsys_playground.git'
repo_dir = 'recsys_playground'
branch_name = '20250908_tiger_dev'

import os
if IN_COLAB:
    if os.path.exists(repo_dir):
      !rm -rf {repo_dir}
    !git clone $repo_url
    %cd $repo_dir
    !git fetch --all
    !git checkout $branch_name || echo 'Branch not found; staying on default.'


In [ ]:
# Runtime & installs
import os, sys, subprocess, torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Install module dependencies (Colab).
!pip -q install -r tiger_semantic_id/requirements.txt

# Make src importable
src_path = os.path.abspath('tiger_semantic_id/src')
if src_path not in sys.path: sys.path.insert(0, src_path)

from tiger_semantic_id.src.utils import set_seed, ensure_dirs, Paths
set_seed(42)
ensure_dirs(Paths.data_dir, Paths.artifacts_dir)


In [ ]:
# Config dataclass
from dataclasses import dataclass

@dataclass
class Config:
    # Dataset selection: "Beauty" or "Video_Games"
    dataset_name: str = 'Beauty'
    min_user_interactions: int = 5
    max_hist_len: int = 20
    embed_model_name: str = 'sentence-t5-base'
    rqvae_latent_dim: int = 32
    rqvae_levels: int = 3
    rqvae_codebook_size: int = 256
    rqvae_beta: float = 0.0025
    rqvae_alpha: float = 0.01
    rqvae_epochs: int = 1000
    rqvae_batch_size: int = 1024
    rqvae_lr: float = 1e-3  # CORRECTED: Reverted to 1e-3 (0.4 was too high)
    seq2seq_d_model: int = 128
    seq2seq_ff: int = 1024
    seq2seq_heads: int = 8  # Changed from 6 to 8 so it divides 128 evenly
    seq2seq_layers_enc: int = 4
    seq2seq_layers_dec: int = 4
    seq2seq_dropout: float = 0.1
    seq2seq_batch_size: int = 256
    seq2seq_steps: int = 20000
    seq2seq_lr: float = 1e-2
    user_vocab_hash: int = 2000
    topk_list: tuple = (5, 10)

cfg = Config()
cfg

In [ ]:
# Fix for Python dict format in metadata files BEFORE any data imports
def _parse_python_dict_lines(path: str):
    """Parse Python dict lines (not JSON) from a gzipped file using ast.literal_eval."""
    import ast
    import gzip

    opener = gzip.open if path.endswith(".gz") else open
    rows = []
    with opener(path, "rt") as f:
        for raw in f:
            try:
                line = raw.strip()
                if line:
                    # Use ast.literal_eval to safely parse Python dict strings
                    data = ast.literal_eval(line)
                    rows.append(data)
            except (ValueError, SyntaxError, MemoryError):
                # Skip malformed lines
                continue
    return rows

# Apply the fix BEFORE importing data functions
from tiger_semantic_id.src import data
data._parse_json_lines = _parse_python_dict_lines
print("✓ Applied Python dict parser fix")

# Now import data functions and download data
from tiger_semantic_id.src.data import DatasetConfig
from tiger_semantic_id.src.utils import Paths

# Create dataset config and get URLs
dataset_cfg = DatasetConfig(dataset_name=cfg.dataset_name)
reviews_url, meta_url = dataset_cfg.get_urls()
reviews_file, meta_file = dataset_cfg.get_filenames()

print(f"Using dataset: {cfg.dataset_name}")
print(f"Reviews: {reviews_url}")
print(f"Metadata: {meta_url}")

!cd /content 2>/dev/null || true
!mkdir -p {Paths.data_dir}
!wget -q -O {Paths.data_dir}/{reviews_file} {reviews_url}
!wget -q -O {Paths.data_dir}/{meta_file} {meta_url}
!gzip -t {Paths.data_dir}/{reviews_file} && gzip -t {Paths.data_dir}/{meta_file} && echo 'gz ok'
!zcat -f {Paths.data_dir}/{reviews_file} | head -n 2
!zcat -f {Paths.data_dir}/{meta_file} | head -n 2

In [ ]:
# Parse and preprocess
import pandas as pd
from tiger_semantic_id.src.data import load_reviews_df, load_meta_df, filter_and_split, build_id_maps, apply_id_maps, save_mappings, DatasetConfig

# Use dataset_cfg created in previous cell
reviews_path = f"{Paths.data_dir}/{reviews_file}"
meta_path = f"{Paths.data_dir}/{meta_file}"

reviews = load_reviews_df(reviews_path)
meta = load_meta_df(meta_path)

# Create DatasetConfig with parameters from cfg
data_cfg = DatasetConfig(
    dataset_name=cfg.dataset_name,
    min_user_interactions=cfg.min_user_interactions,
    max_hist_len=cfg.max_hist_len
)

# Merge item_idx later after mapping
train_df, val_df, test_df = filter_and_split(reviews, data_cfg)
user2id, item2id = build_id_maps([train_df, val_df, test_df])
save_mappings(Paths.artifacts_dir, user2id, item2id)
train_df = apply_id_maps(train_df, user2id, item2id)
val_df = apply_id_maps(val_df, user2id, item2id)
test_df = apply_id_maps(test_df, user2id, item2id)

# Robust merge: ensure metadata has 'item_id' even if source used 'asin'
meta_merge = meta.copy()
print("Meta columns:", meta.columns.tolist())
print("Meta shape:", meta.shape)
items = pd.DataFrame({'item_id': list(item2id.keys()), 'item_idx': list(item2id.values())}).merge(meta, on='item_id', how='left')
print('Shapes:', train_df.shape, val_df.shape, test_df.shape, items.shape)

In [ ]:
# Build item text & embed with Sentence-T5 (GPU/TPU-optimized)
import torch
from tiger_semantic_id.src.embeddings import build_item_text, encode_items
from tiger_semantic_id.src.device_utils import get_device

# Build item texts from metadata
texts = build_item_text(items)
print(f"Built {len(texts)} item text descriptions")

# Auto-detect best device (TPU > GPU > CPU)
device_cfg = get_device("auto")
print(f"\n{'='*60}")
print(f"DEVICE CONFIGURATION")
print(f"{'='*60}")
print(f"Selected device: {device_cfg}")
print(f"Device type: {'TPU' if device_cfg.is_tpu else 'GPU' if device_cfg.is_gpu else 'CPU'}")
print(f"PyTorch device: {device_cfg.device}")

# Adjust batch size based on device
if device_cfg.is_tpu:
    batch_size = 512  # TPUs can handle larger batches
elif device_cfg.is_gpu:
    batch_size = 256  # GPU batch size
else:
    batch_size = 128  # CPU batch size

print(f"Batch size: {batch_size}")
print(f"{'='*60}\n")

# Encode items with device config
item_emb = encode_items(
    texts,
    model_name=cfg.embed_model_name,
    batch_size=batch_size,
    device_config=device_cfg  # Use DeviceConfig instead of device string
)

# Save embeddings to disk
torch.save(item_emb, f"{Paths.artifacts_dir}/item_embeddings.pt")
print(f"\n✅ Saved embeddings to {Paths.artifacts_dir}/item_embeddings.pt")
print(f"Embeddings device: {item_emb.device}")
print(f"Embeddings shape: {item_emb.shape}")

# Debug: Check if the input embeddings themselves are diverse
print(f"\n{'='*60}")
print("INPUT DATA ANALYSIS")
print(f"{'='*60}")
sample_items = item_emb[:10]
print(f"First 2 embeddings identical? {torch.allclose(sample_items[0], sample_items[1])}")
print(f"All 10 embeddings identical? {all(torch.allclose(sample_items[0], sample_items[i]) for i in range(1, 10))}")

# Check actual values
print(f"\nSample embedding values:")
print(f"  Item 0 first 10 dims: {sample_items[0][:10]}")
print(f"  Item 1 first 10 dims: {sample_items[1][:10]}")
print(f"  Item 2 first 10 dims: {sample_items[2][:10]}")

# Check if there's variance within each embedding
print(f"\nInternal variance per embedding:")
for i in range(5):
    print(f"  Item {i} variance: {sample_items[i].var():.6f}")

print(f"\n{'='*60}")
print("✅ Embedding generation complete!")
print(f"{'='*60}")

In [ ]:
# Save all processed data for downstream notebooks
import pickle
import torch
import json

print("=" * 60)
print("SAVING PROCESSED DATA")
print("=" * 60)

# 1. Save DataFrames
print("\n[1] Saving processed DataFrames...")
train_df.to_pickle(f"{Paths.artifacts_dir}/train_df.pkl")
val_df.to_pickle(f"{Paths.artifacts_dir}/val_df.pkl")
test_df.to_pickle(f"{Paths.artifacts_dir}/test_df.pkl")
items.to_pickle(f"{Paths.artifacts_dir}/items.pkl")
print(f"✅ Saved train_df: {train_df.shape}")
print(f"✅ Saved val_df: {val_df.shape}")
print(f"✅ Saved test_df: {test_df.shape}")
print(f"✅ Saved items: {items.shape}")

# 2. Save config for reproducibility
print("\n[2] Saving configuration...")
from dataclasses import asdict
config_dict = asdict(cfg)
with open(f"{Paths.artifacts_dir}/config.json", "w") as f:
    json.dump(config_dict, f, indent=2)
print(f"✅ Saved config.json")

# 3. Save item texts (for reference/debugging)
print("\n[3] Saving item texts...")
with open(f"{Paths.artifacts_dir}/item_texts.json", "w") as f:
    json.dump(texts, f, indent=2)
print(f"✅ Saved item_texts.json: {len(texts)} items")

# 4. Embeddings already saved in T5 cell
print("\n[4] Item embeddings")
print(f"✅ Embeddings already saved: {item_emb.shape}")
print(f"   Location: {Paths.artifacts_dir}/item_embeddings.pt")

print("\n" + "=" * 60)
print("DATA PREPARATION COMPLETE")
print("=" * 60)
print(f"\nSaved to: {Paths.artifacts_dir}/")
print("\nFiles created:")
print("  - train_df.pkl, val_df.pkl, test_df.pkl")
print("  - items.pkl")
print("  - user2id.json, item2id.json")
print("  - config.json")
print("  - item_texts.json")
print("  - item_embeddings.pt")
print("\n✅ Ready for TIGER_SemanticID.ipynb (RQ-VAE training)")
print("=" * 60)